# Employee Attrition Analysis — End-to-End Machine Learning

**Dataset:** IBM HR Analytics Employee Attrition & Performance  
**Goal:** Explore employee attrition and train a reproducible binary classification pipeline.

Run cells from top to bottom in a fresh kernel. Keep `HR-Employee-Attrition.csv` in the same folder as this notebook. This notebook creates:
- `employee_attrition_model.pkl` — complete preprocessing + Logistic Regression pipeline
- `heldout_test_data.csv` — untouched 20% test split (294 rows, 30 features + target)
- `model_evaluation_results.csv` — metrics from the held-out test split
- `model_metadata.json` — package versions and model details

**Important:** The generated model is serialized with scikit-learn 1.8.0. Use the matching `requirements.txt` supplied with this notebook when deploying this exact pickle.

## Part 1 — Imports and reproducibility

In [ ]:
import os
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
import joblib

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, classification_report, confusion_matrix,
    ConfusionMatrixDisplay, RocCurveDisplay
)

warnings.filterwarnings("default")
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (9, 5)

RANDOM_STATE = 10
print("Python environment ready")
print("scikit-learn:", sklearn.__version__)
print("pandas:", pd.__version__)
print("NumPy:", np.__version__)
print("joblib:", joblib.__version__)

## Part 2 — Load the original dataset

In [ ]:
# Put HR-Employee-Attrition.csv beside this notebook.
# The second filename is included for users whose downloaded file retained (1).
possible_paths = [
    Path("HR-Employee-Attrition.csv"),
    Path("HR-Employee-Attrition(1).csv"),
    Path("HR-Employee-Attrition (1).csv"),
]

DATA_PATH = next((path for path in possible_paths if path.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError(
        "Dataset not found. Put HR-Employee-Attrition.csv in the same folder as this notebook."
    )

df = pd.read_csv(DATA_PATH)
print("Loaded:", DATA_PATH.resolve())
print("Shape:", df.shape)
display(df.head())

## Part 3 — Initial data inspection

In [ ]:
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])
display(df.info())

In [ ]:
display(df.describe(include="all").T)

In [ ]:
missing_summary = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_percent": (df.isna().mean() * 100).round(2)
}).sort_values("missing_count", ascending=False)
display(missing_summary)
print("Duplicate rows:", df.duplicated().sum())

In [ ]:
for col in df.columns:
    print(f"{col}: {df[col].nunique()} unique values")

## Part 4 — Exploratory data analysis

In [ ]:
attrition_counts = df["Attrition"].value_counts()
attrition_percent = df["Attrition"].value_counts(normalize=True).mul(100).round(2)
display(pd.DataFrame({"Employees": attrition_counts, "Percentage": attrition_percent}))

In [ ]:
ax = sns.countplot(data=df, x="Attrition", hue="Attrition", palette="Set2", legend=False)
ax.set(title="Employee Attrition Distribution", xlabel="Attrition", ylabel="Employees")
plt.show()

In [ ]:
plt.figure(figsize=(7, 7))
df["Attrition"].value_counts().plot.pie(
    autopct="%1.2f%%", startangle=90, colors=["#66B2FF", "#FF9999"]
)
plt.ylabel("")
plt.title("Employee Attrition Percentage")
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sns.histplot(data=df, x="Age", bins=20, kde=True, ax=axes[0])
axes[0].set_title("Employee Age Distribution")
sns.boxplot(data=df, x="Attrition", y="Age", hue="Attrition", palette="Set2", legend=False, ax=axes[1])
axes[1].set_title("Age by Attrition")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(9, 5))
sns.countplot(data=df, x="Department", hue="Attrition", palette="Set2")
plt.title("Department-wise Employee Attrition")
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(12, 6))
sns.countplot(data=df, y="JobRole", hue="Attrition", palette="Set2")
plt.title("Job Role-wise Employee Attrition")
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
sns.countplot(data=df, x="OverTime", hue="Attrition", palette="Set2", ax=axes[0, 0])
axes[0, 0].set_title("Overtime vs Attrition")
sns.boxplot(data=df, x="Attrition", y="MonthlyIncome", hue="Attrition", palette="Set2", legend=False, ax=axes[0, 1])
axes[0, 1].set_title("Monthly Income vs Attrition")
sns.countplot(data=df, x="JobSatisfaction", hue="Attrition", palette="Set2", ax=axes[1, 0])
axes[1, 0].set_title("Job Satisfaction vs Attrition")
sns.countplot(data=df, x="WorkLifeBalance", hue="Attrition", palette="Set2", ax=axes[1, 1])
axes[1, 1].set_title("Work-Life Balance vs Attrition")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(9, 5))
sns.boxplot(data=df, x="Attrition", y="YearsAtCompany", hue="Attrition", palette="Set2", legend=False)
plt.title("Years at Company vs Attrition")
plt.show()

In [ ]:
department_attrition = (
    df.assign(Left=(df["Attrition"] == "Yes").astype(int))
      .groupby("Department")["Left"]
      .agg(employees="size", attrition_cases="sum", attrition_rate="mean")
      .assign(attrition_rate=lambda x: (x["attrition_rate"] * 100).round(2))
      .sort_values("attrition_rate", ascending=False)
)
display(department_attrition)

## Part 5 — Correlation and grouped analysis

In [ ]:
corr_df = df.copy()
corr_df["Attrition"] = corr_df["Attrition"].map({"No": 0, "Yes": 1})
numeric_df = corr_df.select_dtypes(include=np.number)
correlation_matrix = numeric_df.corr()

plt.figure(figsize=(18, 12))
sns.heatmap(correlation_matrix, cmap="coolwarm", center=0, annot=False)
plt.title("Numerical Feature Correlation Heatmap")
plt.tight_layout()
plt.show()

In [ ]:
attrition_correlations = (
    correlation_matrix["Attrition"]
    .drop("Attrition")
    .sort_values(key=lambda values: values.abs(), ascending=False)
)
display(attrition_correlations.to_frame("Correlation with Attrition"))

In [ ]:
group_metrics = df.groupby("Attrition")[[
    "Age", "MonthlyIncome", "YearsAtCompany",
    "JobSatisfaction", "WorkLifeBalance", "DistanceFromHome"
]].mean().round(2)
display(group_metrics)

## Part 6 — Prepare modeling data

In [ ]:
# Encode target only. Keep the original employee feature values unchanged.
model_df = df.copy()
model_df["Attrition"] = model_df["Attrition"].map({"No": 0, "Yes": 1})

if model_df["Attrition"].isna().any():
    raise ValueError("Unexpected values found in Attrition target.")

# Remove identifier / constant columns, not predictive employee attributes.
columns_to_drop = ["EmployeeNumber", "EmployeeCount", "Over18", "StandardHours"]
model_df = model_df.drop(columns=columns_to_drop)

X = model_df.drop(columns="Attrition")
y = model_df["Attrition"].astype(int)

print("Modeling data:", X.shape)
print("Target:", y.shape)
print("Target distribution:")
display(y.value_counts().rename(index={0: "Stayed", 1: "Left"}))

## Part 7 — Train-test split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("Training target counts:", y_train.value_counts().to_dict())
print("Testing target counts:", y_test.value_counts().to_dict())

assert X_train.shape[0] == 1176
assert X_test.shape[0] == 294
assert X_test.shape[1] == 30
assert y_test.value_counts().to_dict() == {0: 247, 1: 47}

## Part 8 — Preprocessing pipeline

In [ ]:
numeric_features = X_train.select_dtypes(include=np.number).columns.tolist()
categorical_features = X_train.select_dtypes(include=["object", "category"]).columns.tolist()

print("Numerical features:", len(numeric_features))
print("Categorical features:", len(categorical_features))
print("Total features:", len(numeric_features) + len(categorical_features))

numeric_transformer = StandardScaler()
categorical_transformer = OneHotEncoder(handle_unknown="ignore")

preprocessor = ColumnTransformer(
    transformers=[
        ("numerical", numeric_transformer, numeric_features),
        ("categorical", categorical_transformer, categorical_features),
    ],
    remainder="drop"
)

## Part 9 — Train standard and balanced Logistic Regression

In [ ]:
standard_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))
])

balanced_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(
        max_iter=1000, class_weight="balanced", random_state=RANDOM_STATE
    ))
])

standard_model.fit(X_train, y_train)
balanced_model.fit(X_train, y_train)

print("Both Logistic Regression models trained.")

## Part 10 — Cross-validation on training data only

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

standard_cv_f1 = cross_val_score(
    standard_model, X_train, y_train, cv=cv, scoring="f1"
)
balanced_cv_f1 = cross_val_score(
    balanced_model, X_train, y_train, cv=cv, scoring="f1"
)

print("Standard CV F1:", np.round(standard_cv_f1, 4))
print("Standard mean F1:", round(standard_cv_f1.mean(), 4))
print("Balanced CV F1:", np.round(balanced_cv_f1, 4))
print("Balanced mean F1:", round(balanced_cv_f1.mean(), 4))

## Part 11 — Select model and evaluate once on held-out test data

In [ ]:
if standard_cv_f1.mean() >= balanced_cv_f1.mean():
    final_model = standard_model
    final_model_name = "Logistic Regression"
else:
    final_model = balanced_model
    final_model_name = "Balanced Logistic Regression"

# Refit selected pipeline on all training records only.
final_model.fit(X_train, y_train)

y_pred = final_model.predict(X_test)
y_probability = final_model.predict_proba(X_test)[:, 1]

final_metrics = {
    "Accuracy": accuracy_score(y_test, y_pred),
    "Precision": precision_score(y_test, y_pred, zero_division=0),
    "Recall": recall_score(y_test, y_pred, zero_division=0),
    "F1-score": f1_score(y_test, y_pred, zero_division=0),
    "ROC-AUC": roc_auc_score(y_test, y_probability),
}

print("Selected model:", final_model_name)
display(pd.DataFrame({"Metric": final_metrics.keys(), "Score": final_metrics.values()}))
print(classification_report(y_test, y_pred, target_names=["Stayed", "Left"], digits=4))

In [ ]:
cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
disp = ConfusionMatrixDisplay(cm, display_labels=["Stayed", "Left"])
disp.plot(cmap="Blues", values_format="d")
plt.title("Held-out Test Confusion Matrix")
plt.show()

In [ ]:
RocCurveDisplay.from_predictions(y_test, y_probability)
plt.title(f"ROC Curve — {final_model_name}")
plt.show()

## Part 12 — Coefficient analysis

In [ ]:
fitted_preprocessor = final_model.named_steps["preprocessor"]
feature_names = fitted_preprocessor.get_feature_names_out()
coefficients = final_model.named_steps["classifier"].coef_[0]

coefficient_df = pd.DataFrame({
    "Feature": feature_names,
    "Coefficient": coefficients,
    "Absolute Coefficient": np.abs(coefficients)
}).sort_values("Absolute Coefficient", ascending=False)

display(coefficient_df.head(20))

In [ ]:
top_coefficients = coefficient_df.head(15).sort_values("Coefficient")
plt.figure(figsize=(10, 7))
sns.barplot(data=top_coefficients, x="Coefficient", y="Feature", hue="Coefficient", palette="coolwarm", legend=False)
plt.axvline(0, color="black", linewidth=1)
plt.title("Top Logistic Regression Coefficients")
plt.tight_layout()
plt.show()

## Part 13 — Export the exact held-out dataset

In [ ]:
# This is the exact X_test/y_test split used above, not the full dataset.
heldout_test_data = X_test.copy()
heldout_test_data["Attrition"] = y_test.to_numpy()

heldout_test_data.to_csv("heldout_test_data.csv", index=False)

print("Exported heldout_test_data.csv")
print("Shape:", heldout_test_data.shape)
print("Target distribution:", heldout_test_data["Attrition"].value_counts().sort_index().to_dict())

assert heldout_test_data.shape == (294, 31)
assert heldout_test_data["Attrition"].value_counts().to_dict() == {0: 247, 1: 47}

## Part 14 — Save the complete model pipeline

In [ ]:
# The pickle includes preprocessing AND classifier.
# Do not save only the LogisticRegression estimator.
joblib.dump(final_model, "employee_attrition_model.pkl")

# Save evaluation metrics and reproducibility metadata.
pd.DataFrame({
    "Metric": list(final_metrics.keys()),
    "Score": list(final_metrics.values())
}).to_csv("model_evaluation_results.csv", index=False)

metadata = {
    "model_file": "employee_attrition_model.pkl",
    "model_type": final_model_name,
    "scikit_learn_version": sklearn.__version__,
    "pandas_version": pd.__version__,
    "numpy_version": np.__version__,
    "joblib_version": joblib.__version__,
    "input_feature_count": int(X.shape[1]),
    "feature_columns": X.columns.tolist(),
    "target_mapping": {"No": 0, "Yes": 1},
    "train_test_split": {
        "test_size": 0.20,
        "random_state": RANDOM_STATE,
        "stratify": True
    },
    "cross_validation_mean_f1": {
        "standard": float(standard_cv_f1.mean()),
        "balanced": float(balanced_cv_f1.mean())
    },
    "heldout_metrics": {key: float(value) for key, value in final_metrics.items()}
}
with open("model_metadata.json", "w") as file:
    json.dump(metadata, file, indent=2)

print("Saved employee_attrition_model.pkl")
print("Saved heldout_test_data.csv")
print("Saved model_evaluation_results.csv")
print("Saved model_metadata.json")

## Part 15 — Verify the saved model and held-out evaluation

In [ ]:
loaded_model = joblib.load("employee_attrition_model.pkl")

# Verify predictions from the reloaded artifact.
reloaded_predictions = loaded_model.predict(X_test)
reloaded_probabilities = loaded_model.predict_proba(X_test)[:, 1]

assert np.array_equal(reloaded_predictions, y_pred)
assert np.allclose(reloaded_probabilities, y_probability)

print("Model reload verification passed.")
print("Model file exists:", Path("employee_attrition_model.pkl").exists())
print("Model file size (bytes):", Path("employee_attrition_model.pkl").stat().st_size)
print("Held-out CSV exists:", Path("heldout_test_data.csv").exists())
print("Reloaded model accuracy:", round(accuracy_score(y_test, reloaded_predictions), 4))

## Final notes

- Upload `app.py`, `employee_attrition_model.pkl`, `heldout_test_data.csv`, `HR-Employee-Attrition.csv`, and `requirements.txt` to the same GitHub repository branch used by Streamlit Cloud.
- The app should load the pickle with `joblib.load()` and evaluate it using the held-out CSV. The held-out CSV contains the target column `Attrition`; pass only the other 30 columns to `model.predict()` and `model.predict_proba()`.
- Keep the exact scikit-learn version in `requirements.txt` aligned with the version printed in Part 1.
- The reported test metrics are from this fixed split. They are not evidence of causal relationships or guaranteed performance on future employees.